# YgoMaster-PackBuilder — Policy Conversion Tests

Notebook di test per il converter `PoliciesConfig -> list[OpeningPolicy]`.

Verifica caricamento, validation, conversione, tipi domain, valori di
`configuration-a` e indipendenza tra DTO e oggetti del domain.


## 1. Ambiente e root della repository

In [1]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository."
    )

policies_path = repo_root / "config" / "policies.yaml"
assert policies_path.exists(), f"Policies config non trovato: {policies_path}"

print("Repository root:", repo_root)
print("Policies path:", policies_path)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder
Policies path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\policies.yaml


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.policies import (
    convert_policies_to_domain,
    load_policies_config,
)

from ygomaster_packbuilder.validation import validate_policies_config

from ygomaster_packbuilder.domain import (
    OpeningPolicy,
    RarityDistribution,
    TenPackGuarantee,
    UltraRarePity,
)

print("Imports OK")

Imports OK


## 3. Caricamento e validation

In [3]:
config = load_policies_config(policies_path)
report = validate_policies_config(config)

assert report.is_valid, [
    (issue.code, issue.path, issue.message)
    for issue in report.errors
]

print("Policies config is valid")

Policies config is valid


## 4. Conversione verso il domain

In [4]:
domain_policies = convert_policies_to_domain(config)

assert isinstance(domain_policies, list)
assert len(domain_policies) == len(config.policies)

print([policy.key for policy in domain_policies])

['configuration-a']


## 5. Tipi domain

In [5]:
for policy in domain_policies:
    assert isinstance(policy, OpeningPolicy)
    assert isinstance(policy.normal_slot_distribution, RarityDistribution)
    assert isinstance(policy.ten_pack_guarantee, TenPackGuarantee)
    assert isinstance(
        policy.ten_pack_guarantee.final_slot_distribution,
        RarityDistribution,
    )
    assert isinstance(policy.ultra_rare_pity, UltraRarePity)
    assert isinstance(
        policy.ultra_rare_pity.guaranteed_slot_distribution,
        RarityDistribution,
    )

print("All converted objects use domain types")

All converted objects use domain types


## 6. Configuration A

In [6]:
configuration_a = next(
    (p for p in domain_policies if p.key == "configuration-a"),
    None,
)

assert configuration_a is not None
configuration_a

OpeningPolicy(key='configuration-a', normal_slot_distribution=RarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5), ten_pack_guarantee=TenPackGuarantee(enabled=True, final_slot_distribution=RarityDistribution(normal=0.0, rare=0.0, super_rare=80.0, ultra_rare=20.0)), ultra_rare_pity=UltraRarePity(enabled=True, guaranteed_slot_distribution=RarityDistribution(normal=0.0, rare=0.0, super_rare=0.0, ultra_rare=100.0)))

## 7. Distribuzione normale

In [7]:
d = configuration_a.normal_slot_distribution

assert d.normal == 55.0
assert d.rare == 35.0
assert d.super_rare == 7.5
assert d.ultra_rare == 2.5

print(d)

RarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5)


## 8. Ten-pack guarantee

In [8]:
g = configuration_a.ten_pack_guarantee

assert g.enabled is True
assert g.final_slot_distribution.normal == 0.0
assert g.final_slot_distribution.rare == 0.0
assert g.final_slot_distribution.super_rare == 80.0
assert g.final_slot_distribution.ultra_rare == 20.0

print(g)

TenPackGuarantee(enabled=True, final_slot_distribution=RarityDistribution(normal=0.0, rare=0.0, super_rare=80.0, ultra_rare=20.0))


## 9. Ultra Rare pity

In [9]:
p = configuration_a.ultra_rare_pity

assert p.enabled is True
assert p.guaranteed_slot_distribution.normal == 0.0
assert p.guaranteed_slot_distribution.rare == 0.0
assert p.guaranteed_slot_distribution.super_rare == 0.0
assert p.guaranteed_slot_distribution.ultra_rare == 100.0

print(p)

UltraRarePity(enabled=True, guaranteed_slot_distribution=RarityDistribution(normal=0.0, rare=0.0, super_rare=0.0, ultra_rare=100.0))


## 10. DTO e domain object sono istanze distinte

In [10]:
source_policy = next(
    p for p in config.policies if p.key == "configuration-a"
)

assert configuration_a is not source_policy
assert configuration_a.normal_slot_distribution is not source_policy.normal_slot_distribution
assert configuration_a.ten_pack_guarantee is not source_policy.ten_pack_guarantee
assert configuration_a.ultra_rare_pity is not source_policy.ultra_rare_pity

print("Independent instances OK")

Independent instances OK


## 11. Mutazioni successive dei DTO non modificano il domain

In [11]:
test_config = deepcopy(config)
converted = convert_policies_to_domain(test_config)

domain_policy = next(
    p for p in converted if p.key == "configuration-a"
)
source_policy = next(
    p for p in test_config.policies if p.key == "configuration-a"
)

original = domain_policy.normal_slot_distribution.normal
source_policy.normal_slot_distribution.normal = 1.0

assert domain_policy.normal_slot_distribution.normal == original

print("Domain object remains unchanged")

Domain object remains unchanged


## 12. Ordine preservato

In [12]:
source_keys = [p.key for p in config.policies]
domain_keys = [p.key for p in domain_policies]

assert domain_keys == source_keys

print(domain_keys)

['configuration-a']


## 13. Conversione di più policies

In [13]:
test_config = deepcopy(config)

extra_policy = deepcopy(test_config.policies[0])
extra_policy.key = "configuration-b"
test_config.policies.append(extra_policy)

converted = convert_policies_to_domain(test_config)
keys = [p.key for p in converted]

assert "configuration-a" in keys
assert "configuration-b" in keys

print(keys)

['configuration-a', 'configuration-b']
